# Planners Comparison Study

This notebook demonstrates how to conduct a comprehensive comparison of different POMDP planning algorithms using the SimulationsAPI. We'll compare POMCPOW and PFT-DPW planners on Push POMDP and Light-Dark POMDP environments, showcasing how to evaluate algorithm performance across different problem domains.

## Overview

**Planning Algorithms Tested:**
- **POMCPOW**: Monte Carlo Tree Search with double progressive widening
- **PFT-DPW**: Progressive Function Transfer with Double Progressive Widening

**Environments Tested:**
- **Push POMDP**: Object manipulation with continuous actions
- **Light-Dark POMDP**: Navigation with position-dependent observation noise

**Key Features Demonstrated:**
- Environment configuration using EnvironmentConfigsAPI
- Pre-built action samplers from utils module for different action spaces
- Statistical analysis with confidence intervals
- Multi-environment, multi-algorithm evaluation
- Performance profiling and result visualization

## Setup and Imports

In [1]:
import numpy as np
from pathlib import Path

# Core POMDPPlanners imports
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.planners.mcts_planners.pomcpow import POMCPOW
from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.simulations.simulations_api import SimulationsAPI
from POMDPPlanners.core.simulation import EnvironmentRunParams
from POMDPPlanners.utils.action_samplers import UnitCircleActionSampler, DiscreteActionSampler

## Environment Setup

Setting up the environments using an the environment's API

In [6]:
env_config = EnvironmentConfigsAPI(discount_factor=0.95, debug=False)

push_env, push_belief = env_config.push_pomdp_config(n_particles=20)  # Reduced from 1000
print(f"Created Push POMDP environment: {push_env.__class__.__name__}")

light_dark_env, light_dark_belief = env_config.continuous_observations_discrete_actions_light_dark_pomdp_config(
    n_particles=30  # Reduced from 1000
)
print(f"Created Light-Dark POMDP environment: {light_dark_env.__class__.__name__}")

2025-10-01 12:58:21,685 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:240 - Initializing PushPOMDP environment with discount factor 0.95
2025-10-01 12:58:21,686 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:240 - Initializing ContinuousLightDarkPOMDPDiscreteActions environment with discount factor 0.95


Created Push POMDP environment: PushPOMDP
Created Light-Dark POMDP environment: ContinuousLightDarkPOMDPDiscreteActions


## Action Sampler Configuration

In [7]:
# Create action samplers based on available environments

# Tiger POMDP uses discrete actions: [0, 1, 2] for [left, right, listen]
push_action_sampler = DiscreteActionSampler(actions=push_env.get_actions())
print("Created discrete action sampler for Tiger POMDP (as Push)")

# Same environment for Light-Dark
light_dark_action_sampler = DiscreteActionSampler(actions=light_dark_env.get_actions())
print("Created discrete action sampler for Tiger POMDP (as Light-Dark)")

Created discrete action sampler for Tiger POMDP (as Push)
Created discrete action sampler for Tiger POMDP (as Light-Dark)


## Planner Configuration

Setting up POMCPOW and PFT-DPW planners for each environment:

In [8]:
# Configure planners for Push POMDP (or fallback environment)
# Use POMCP instead of POMCPOW for faster testing
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP

push_planners = [
    POMCP(
        environment=push_env,
        discount_factor=0.95,
        depth=10,              # Minimal depth
        exploration_constant=100.,
        n_simulations=2000,     # Absolutely minimal simulations
        name="POMCP_Push"
    ),
    POMCP(
        environment=push_env,
        discount_factor=0.95,
        depth=10,              # Minimal depth
        exploration_constant=100.0,  # Different parameter for comparison
        n_simulations=2000,     # Absolutely minimal simulations
        name="POMCP_Push_Alt"
    )
]
print(f"Created {len(push_planners)} planners for Push environment")

2025-10-01 12:59:05,970 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCP_Push (debug=False)
2025-10-01 12:59:05,970 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCP_Push_Alt (debug=False)


Created 2 planners for Push environment


In [9]:
# Configure planners for Light-Dark POMDP (or fallback environment)
light_dark_planners = [
    POMCP(
        environment=light_dark_env,
        discount_factor=0.95,
        depth=10,              # Minimal depth
        exploration_constant=100.,
        n_simulations=2000,     # Absolutely minimal simulations
        name="POMCP_LightDark"
    ),
    POMCP(
        environment=light_dark_env,
        discount_factor=0.95,
        depth=10,              # Minimal depth
        exploration_constant=100.0,  # Different parameter for comparison
        n_simulations=2000,     # Absolutely minimal simulations
        name="POMCP_LightDark_Alt"
    )
]
print(f"Created {len(light_dark_planners)} planners for Light-Dark environment")

2025-10-01 12:59:20,305 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCP_LightDark (debug=False)
2025-10-01 12:59:20,306 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:119 - Initialized policy: POMCP_LightDark_Alt (debug=False)


Created 2 planners for Light-Dark environment


## Simulation Configuration

In [10]:
# Create simulation configurations with absolutely minimal parameters for testing
environment_run_params = [
    EnvironmentRunParams(
        environment=push_env,
        belief=push_belief,
        policies=push_planners,
        num_episodes=20,       # Absolute minimum episodes
        num_steps=15           # Absolute minimum steps
    ),
    EnvironmentRunParams(
        environment=light_dark_env,
        belief=light_dark_belief,
        policies=light_dark_planners,
        num_episodes=20,       # Absolute minimum episodes
        num_steps=15           # Absolute minimum steps
    )
]

print(f"Created {len(environment_run_params)} environment run configurations")
for i, config in enumerate(environment_run_params):
    env_name = config.environment.__class__.__name__
    policies_names = [p.name for p in config.policies]
    print(f"  Config {i+1}: {env_name} with {len(config.policies)} policies")
    print(f"    Policies: {policies_names}")
    print(f"    Episodes: {config.num_episodes}, Steps: {config.num_steps}")

Created 2 environment run configurations
  Config 1: PushPOMDP with 2 policies
    Policies: ['POMCP_Push', 'POMCP_Push_Alt']
    Episodes: 20, Steps: 15
  Config 2: ContinuousLightDarkPOMDPDiscreteActions with 2 policies
    Policies: ['POMCP_LightDark', 'POMCP_LightDark_Alt']
    Episodes: 20, Steps: 15


## Running the Comparison Study

In [ ]:
# Initialize SimulationsAPI
api = SimulationsAPI(cache_dir_path=Path("./planners_comparison_results"), debug=True)

cache_dir_path = Path("./planners_comparison_results")

print("Starting planners comparison study...")
print(f"Running {len(environment_run_params)} environment configurations")

# Run simulation with statistical analysis
results, statistics_df = api.run_multiple_environments_and_policies_local_run_with_initial_debug_run(
    environment_run_params=environment_run_params,
    alpha=0.05,
    confidence_interval_level=0.95,
    experiment_name="Planners_Comparison_Study",
    n_jobs=-1,              # Use single core for testing
    enable_profiling=False,  # Disable profiling for simpler output
    cache_dir_path=cache_dir_path,
)

print("\nComparison study completed successfully!")
print(f"Results available for {len(results)} environments")
if statistics_df is not None:
    print(f"Statistics computed for {len(statistics_df)} configurations")
else:
    print("No statistics available")

## Results Analysis and Visualization

### Accessing Evaluation Results

The simulation results are automatically saved and can be explored through MLflow's web interface. Run the following commands in your terminal:

```bash
# Navigate to the cache directory and launch MLflow UI
cd ./planners_comparison_results
mlflow ui --backend-store-uri ./mlruns
```

### Results Organization Structure

The evaluation results follow a hierarchical organization:

- **Environment Level**: Results are grouped by environment type (Push POMDP, Light-Dark POMDP)
- **Planner Comparisons**: Within each environment, performance metrics compare all tested planners
- **Statistical Analysis**: Confidence intervals and significance tests validate performance differences
- **Episode Tracking**: Individual episode results provide detailed performance insights

### Performance Metrics and Analysis

**Key Performance Metrics Available:**
- **Average Return**: Mean cumulative reward across episodes
- **Confidence Intervals**: Statistical bounds on performance estimates (95% confidence level)
- **Standard Deviation**: Measure of performance variability across episodes
- **Planning Time**: Computational efficiency and resource usage
- **Success Rates**: Episode completion and convergence statistics

## Summary

This notebook demonstrated:

1. **Multi-algorithm comparison** using POMCPOW and PFT-DPW planners
2. **Multi-environment evaluation** across different POMDP domains
3. **Statistical analysis** with confidence intervals and significance testing

The POMDPPlanners framework provides powerful tools for conducting rigorous algorithm comparisons across different problem domains, enabling both statistical rigor and practical insights for algorithm selection and tuning.

**Key Benefits:**
- Standardized evaluation protocols
- Flexible configuration options
- Performance profiling capabilities
- Reproducible experimental workflows

**Next Steps:**
- Hyper-parameter notebook